
# Đồ án Python (OOP) – Tìm đường đi ngắn nhất bằng Dijkstra
**Mục tiêu:** Xây dựng chương trình *hướng đối tượng* để tìm đường đi ngắn nhất giữa hai đỉnh bất kỳ trong **đồ thị vô hướng có trọng số**, đọc dữ liệu từ file **`Graph.csv`** (mỗi dòng: `v_from,v_to,weight`).


In [1]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: console dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [2]:
import csv
import heapq
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Button, Layout
import networkx as nx
from IPython.display import display

In [3]:
demo_csv = """v_from,v_to,weight
A,B,4
A,C,2
B,C,5
B,D,5
C,D,3
D,E,1
C,E,6
"""
with open('Graph.csv','w', encoding='utf-8') as f:
    f.write(demo_csv)
print("Đã tạo Graph.csv mẫu trong thư mục làm việc.")


Đã tạo Graph.csv mẫu trong thư mục làm việc.


In [4]:
class Node:
  def __init__(self, name):
    self.name = name
  def __eq__(self, other):
    return self.name == other.name
  def __hash__(self):
    return hash(self.name)
  def __repr__(self):
    return f"Node({self.name})"


class Edge:
  def __init__(self, v_from, v_to, weight):
    self.v_from = v_from
    self.v_to = v_to
    self.weight = weight
  def other(self, node):
    if node == self.v_from:
        return self.v_to
    elif node == self.v_to:
        return self.v_from
    else:
        raise ValueError(f"{node} không thuộc cạnh này")
  def __repr__(self):
    return f"Edge({self.v_from.name} - {self.v_to.name}, w={self.weight})"


class Graph:
  def __init__(self):
    self.adj = {}
    self.nodes = {}
    self.edges = []
  def add_edge(self, u_name, v_name, weight):
    u = self.nodes.setdefault(u_name, Node(u_name))
    v = self.nodes.setdefault(v_name, Node(v_name))
    edge = Edge(u, v, weight)
    self.edges.append(edge)


    if u_name not in self.adj:
      self.adj[u_name] = []
    if v_name not in self.adj:
      self.adj[v_name] = []


    if (v_name, weight) not in self.adj[u_name]:
      self.adj[u_name].append((v_name, weight))
    if (u_name, weight) not in self.adj[v_name]:
      self.adj[v_name].append((u_name, weight))

  def load_from_csv(self, filename):
    with open(filename, 'r', encoding='utf-8') as f:
      reader = csv.DictReader(f)
      for row in reader:
        u = row['v_from'].strip()
        v = row['v_to'].strip()
        w = float(row['weight'])
        self.add_edge(u, v, w)

  def get_neighbors(self, node_name):
    return self.adj.get(node_name, [])

  def vertices(self):
    return list(self.adj.keys())

  def __repr__(self):
    s = "Đồ thị vô hướng có trọng số:\n"
    for u in self.adj:
      s += f"{u} → {self.adj[u]}\n"
    return s


In [1]:
class Dijkstra:
    def __init__(self, graph):
        self.graph = graph
        self.steps = []

    def shortest_path(self, start):
        self.steps = []
        dist = {v: float('inf') for v in self.graph.vertices()}
        dist[start] = 0
        prev = {v: None for v in self.graph.vertices()}

        pq = [(0, start)]
        settled = set()

        self.steps.append({
            "t": len(self.steps) + 1,
            "type": "init",
            "current": start,
            "settled": [],
            "dist": dict(dist),  
            "path": None
        })

        while pq:
            current_dist, u = heapq.heappop(pq)
            if u in settled:
                continue
            settled.add(u)

            self.steps.append({
                "t": len(self.steps) + 1,
                "type": "settle",
                "current": u,
                "settled": list(settled),
                "dist": dict(dist),
                "path": self.get_path(prev, start, u)
            })
            for v, weight in self.graph.get_neighbors(u):
                if v in settled:
                    continue
                new_dist = current_dist + weight
                updated = False
                if new_dist < dist[v]:
                    dist[v] = new_dist
                    prev[v] = u
                    heapq.heappush(pq, (new_dist, v))
                    updated = True

                self.steps.append({
                    "t": len(self.steps) + 1,
                    "type": "edge_check",
                    "current": u,
                    "neighbor": v,
                    "weight": weight,
                    "new_dist": new_dist,
                    "updated": updated,
                    "settled": list(settled),
                    "dist": dict(dist),
                    "path": self.get_path(prev, start, u)
                })
        return dist, prev

    def get_path(self, prev, start, end):
        path = []
        node = end
        while node is not None:
            path.insert(0, node)
            node = prev[node]
        if len(path) == 0 or path[0] != start:
            return None
        return path


    def find_shortest_path(self, start, end):
      dist, prev = self.shortest_path(start)
      if dist[end] == float('inf'):
          print(f"\nHai đỉnh {start} và {end} không có đường đi.")
          return float('inf'), None
      else:
          path = self.get_path(prev, start, end)
          print(f"\nKhoảng cách ngắn nhất: {dist[end]}")
          print(f"Đường đi: {path}")
          return dist[end], path


In [ ]:
class Animate:
    def __init__(self, steps, graph, shortest_path=None):
        self.steps = steps
        self.graph = graph
        self.shortest_path = shortest_path
        self.G = nx.Graph()
        for u in graph.adj:
            for v, w in graph.adj[u]:
                self.G.add_edge(u, v, weight=w)
        self.pos = nx.kamada_kawai_layout(self.G)
        self.edge_labels = nx.get_edge_attributes(self.G, 'weight')

    def render_step(self, k, dist=None):
        step = self.steps[k]
        fig, ax = plt.subplots(figsize=(12, 10))
        ax.set_title(f"Bước thứ {step.get('t')}", loc='left')

        current = step.get("current")
        settled = set(step.get("settled", []))
        path = step.get("path")

        frontier = set()
        if step.get("type") == "edge_check":
         # Khi xét edge, frontier là tất cả dist < inf mà chưa settled
         frontier = {node for node, d in step.get("dist", {}).items() if d != float('inf') and node not in settled}
        else:
         # Khi settle, frontier là các đỉnh chưa settled có dist < inf
         frontier = {node for node, d in step.get("dist", {}).items() if d != float('inf') and node not in settled}

        node_colors, node_sizes = [], []


        for n in self.G.nodes():
            if n == current:
                node_colors.append("orange")
                node_sizes.append(1400)
            elif n in settled:
                node_colors.append("lightgreen")
                node_sizes.append(1100)
            elif n in frontier:
                node_colors.append("lightyellow")
                node_sizes.append(1000)
            elif path and n in path:
                node_colors.append("yellow")
                node_sizes.append(1100)
            else:
                node_colors.append("lightgray")
                node_sizes.append(900)


        current_dist = step.get("dist", dist)

        if current_dist:
            labels = {}
            for node in self.G.nodes():
                d = current_dist.get(node, float('inf'))
                labels[node] = f"{node}\n({d if d != float('inf') else '∞'})"
        else:
            labels = {node: str(node) for node in self.G.nodes()}


        nx.draw_networkx_nodes(self.G, self.pos, node_color=node_colors, node_size=node_sizes, ax=ax)
        nx.draw_networkx_labels(self.G, self.pos, labels=labels, ax=ax)

        edge_colors, edge_widths = [], []


        path_edges = []
        if path and len(path) > 1:
            path_edges = [(path[i], path[i + 1]) for i in range(len(path) - 1)]


        considered_edge = None
        if step.get("type") == "edge_check":
            considered_edge = (step["current"], step["neighbor"])

        for (u, v) in self.G.edges():
            color, width = "gray", 1.5

            if considered_edge:
                u_ce, v_ce = considered_edge
                if (u == u_ce and v == v_ce) or (u == v_ce and v == u_ce):
                    color, width = "red", 3.5
            elif k == len(self.steps) - 1 and self.shortest_path:
               path_edges = list(zip(self.shortest_path, self.shortest_path[1:]))
               if (u, v) in path_edges or (v, u) in path_edges:
                 color, width = "purple", 3

            edge_colors.append(color)
            edge_widths.append(width)

        nx.draw_networkx_edges(self.G, self.pos, edge_color=edge_colors, width=edge_widths, ax=ax)
        nx.draw_networkx_edge_labels(self.G, self.pos, edge_labels=self.edge_labels, ax=ax)

        # Kiểm tra trường hợp không có đường đi
        if k == len(self.steps) - 1:
            if not self.shortest_path or len(self.shortest_path) <= 1:
                nodes = list(self.G.nodes())
                if len(nodes) >= 2:
                    start, end = nodes[0], nodes[-1]
        elif k == len(self.steps) - 1 and self.shortest_path:
            path_edges = list(zip(self.shortest_path, self.shortest_path[1:]))
            nx.draw_networkx_edges(self.G, self.pos, edgelist=path_edges, edge_color="blue", width=4, ax=ax)
            nx.draw_networkx_nodes(self.G, self.pos, nodelist=self.shortest_path,
                                   node_color="skyblue", node_size=1000, ax=ax)
            ax.set_title("Đường đi ngắn nhất", loc="center")

        ax.axis("off")
        plt.tight_layout()
        return fig

In [7]:
class AnimateController:
    def __init__(self, animator, dist=None):
        self.anim = animator
        self.dist = dist
        self.n_steps = len(animator.steps)

        self.slider = widgets.IntSlider(value=0, min=0, max=self.n_steps - 1, step=1,
                                        description="Step", layout=Layout(width='50%', height='80px'))
        self.play = widgets.Play(interval=800, min=0, max=self.n_steps - 1, step=1)
        widgets.jslink((self.play, 'value'), (self.slider, 'value'))

        self.out = widgets.Output()
        self.slider.observe(self._update, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.play, self.slider]),
            self.out
        ])
        self._update({'new': 0})

    def _update(self, change):
        k = change['new']
        with self.out:
            self.out.clear_output(wait=True)
            fig = self.anim.render_step(k, dist=self.dist)
            display(fig)
            plt.close(fig)

    def display(self):
        display(self.ui)

In [8]:
filename = 'Graph.csv'
g = Graph()
g.load_from_csv(filename)
print(g)

start = input("Nhập đỉnh bắt đầu: ").strip()
end = input("Nhập đỉnh kết thúc: ").strip()

algo = Dijkstra(g)
dist, path = algo.find_shortest_path(start, end)
anim = Animate(algo.steps, g, shortest_path=path)
ctl = AnimateController(anim, dist=algo.shortest_path(start)[0])
ctl.display()


Đồ thị vô hướng có trọng số:
A → [('B', 4.0), ('C', 2.0)]
B → [('A', 4.0), ('C', 5.0), ('D', 5.0)]
C → [('A', 2.0), ('B', 5.0), ('D', 3.0), ('E', 6.0)]
D → [('B', 5.0), ('C', 3.0), ('E', 1.0)]
E → [('D', 1.0), ('C', 6.0)]


Khoảng cách ngắn nhất: 6.0
Đường đi: ['A', 'C', 'D', 'E']


In [9]:
#case khác
demo_csv = """v_from,v_to,weight
A,B,1
A,C,2
B,D,10
B,E,2
C,E,3
C,D,8
E,F,1
D,F,2
"""
with open('Graph2.csv','w', encoding='utf-8') as f:
    f.write(demo_csv)
print("Đã tạo Graph2.csv mẫu trong thư mục làm việc.")

Đã tạo Graph2.csv mẫu trong thư mục làm việc.


In [ ]:
filename = 'Graph2.csv'
g = Graph()
g.load_from_csv(filename)
print(g)

start = input("Nhập đỉnh bắt đầu: ").strip()
end = input("Nhập đỉnh kết thúc: ").strip()

algo = Dijkstra(g)
dist, path = algo.find_shortest_path(start, end)

print(algo.shortest_path(start)[0])

anim = Animate(algo.steps, g, shortest_path=path)
ctl = AnimateController(anim, dist=algo.shortest_path(start)[0])
ctl.display()

Đồ thị vô hướng có trọng số:
A → [('B', 1.0), ('C', 2.0)]
B → [('A', 1.0), ('D', 10.0), ('E', 2.0)]
C → [('A', 2.0), ('E', 3.0), ('D', 8.0)]
D → [('B', 10.0), ('C', 8.0), ('F', 2.0)]
E → [('B', 2.0), ('C', 3.0), ('F', 1.0)]
F → [('E', 1.0), ('D', 2.0)]



In [ ]:
#case khác
demo_csv = """v_from,v_to,weight
A,B,3
A,C,4
B,C,2
B,D,2
C,E,3
D,E,2
D,F,5
E,F,3
"""
with open('Graph3.csv','w', encoding='utf-8') as f:
    f.write(demo_csv)
print("Đã tạo Graph3.csv mẫu trong thư mục làm việc.")

In [ ]:
filename = 'Graph3.csv'
g = Graph()
g.load_from_csv(filename)
print(g)

start = input("Nhập đỉnh bắt đầu: ").strip()
end = input("Nhập đỉnh kết thúc: ").strip()

algo = Dijkstra(g)
dist, path = algo.find_shortest_path(start, end)
anim = Animate(algo.steps, g, shortest_path=path)
ctl = AnimateController(anim, dist=algo.shortest_path(start)[0])
ctl.display()

In [ ]:
#case khác
demo_csv = """v_from,v_to,weight
A,B,1
A,C,2
B,D,4
B,E,2
C,E,3
C,D,3
E,F,1
D,F,2
G,H,1
"""
with open('Graph4.csv','w', encoding='utf-8') as f:
    f.write(demo_csv)
print("Đã tạo Graph4.csv mẫu trong thư mục làm việc.")

In [ ]:
# trường hợp không có đường đi
filename = 'Graph4.csv'
g = Graph()
g.load_from_csv(filename)
print(g)

start = input("Nhập đỉnh bắt đầu: ").strip()
end = input("Nhập đỉnh kết thúc: ").strip()
algo = Dijkstra(g)
dist, path = algo.find_shortest_path(start, end)
anim = Animate(algo.steps, g, shortest_path=path)
ctl = AnimateController(anim, dist=algo.shortest_path(start)[0])
ctl.display()